# Universal Imports

### Libraries

In [ ]:
import pandas as pd
from sqlalchemy import text
from itertools import product
import plotly.express as px
import matplotlib.pyplot as plt
from plotnine import ggplot, aes, geom_bar, geom_text, position_dodge, scale_fill_brewer, scale_y_continuous, labs, theme_seaborn, theme, scale_fill_manual, element_line, geom_hline, geom_line, \
geom_point, scale_x_datetime, element_text, element_blank, geom_boxplot, stat_summary, scale_color_manual, element_rect, guides, guide_legend


import os
import datetime as dt
from dateutil.relativedelta import relativedelta
from zoneinfo import ZoneInfo
import io
from pathlib import Path

### Variables

In [ ]:
# Engine
from core.functions import engine, workout_theme


# Defining the save directory for output graphs
repo_path = None

for x in range(5):
    # Use .parents to go up a level in the file path, x is a number list how far back we want to go 0 = parent, 1 = grandparent, and so on
    candidate_path = Path.cwd().parents[x]

    # For each candidate path we're checking if main.py exists because that indicates we're at the main repo
    if (candidate_path / "main.py").is_file():
        repo_path = candidate_path
        print(f"Working directory found: '{repo_path}' ")
        break  # exits the outer loop too

## Raising an error here if we can't find apple folder using main.py indicator
if repo_path is None:
    raise FileNotFoundError("Could not find main.py")

## Building the path to where we want to save the charts within the static folder 
output_path = repo_path / "static" / "charts"
### printing output path as a check 
print(f"Here's the path to the charts folder: '{output_path}'")


# Date ranges for filtering data
today = dt.datetime.now(tz=ZoneInfo("US/Eastern"))
l_3_m = today - relativedelta(weeks=14)
l_7_m = today - relativedelta(months=7)
l_1_y = today - relativedelta(years=1)
l_2_w = today - relativedelta(weeks=2)

### Functions

In [ ]:
# resuable function to save plotnine graphs as pngs
def save_plot(plot, filename: str):
    plot.save(
        filename,
        width=10,
        height=5,
        dpi=300,
        verbose=False,
    )



# Function that fills in missing data for each grouped by df 
def fill_missing_combinations(
    original_df,
    aggregated_df,
    time_col,
    category_col,
    value_cols,
    time_filter=None,
    category_values=None):

    # Step 1: Filter time periods if needed
    if time_filter:
        time_values = original_df[time_filter(original_df)].loc[:, time_col].unique()
    else:
        time_values = original_df[time_col].unique()
    # Step 2: Get all categories
    if category_values:
        category_values = category_values
    else:
        category_values = aggregated_df[category_col].unique()
    # Step 3: Create full index
    full_index = pd.DataFrame(
        product(category_values, time_values),
        columns=[category_col, time_col])
    # Step 4: Merge with aggregated data
    filled_df = pd.merge(full_index, aggregated_df, on=[category_col, time_col], how='left')
    # Step 5: Fill NaNs with 0 for specified value columns
    for col in value_cols:
        filled_df[col] = filled_df[col].fillna(0)

    return filled_df.sort_values(by=[time_col, category_col]).reset_index(drop=True)

### Cell skip logic

In [ ]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    """A cell magic to skip the execution of a cell with '%%skip'"""
    return

## Reading in Apple Workouts db from SQLite

In [ ]:
# Read in the data and parse datetime columns
with engine.connect() as connection:
    aw_all = pd.read_sql_query(
        text("""
             SELECT *
             FROM apple_workouts
             """), connection)
        #dtype={"workout_id": "Int64"}) # Converting workout_id to Int64 upon entry, will need again if workout_id includes blanks
    
# Converting the string datetime variable from db to with timezone normalization to UTC then converting to EST
# This is needed because of combination of EST and EDT in the data 
# I could do this with a parse dates function in pd.read_sql_query() function above
date_cols = ["StartDate"]
for col in date_cols:
    aw_all[col] = (
        pd.to_datetime(aw_all[col])
        .dt.tz_localize("UTC")
        .dt.tz_convert("US/Eastern"))

### Adding in categorical variables for month and week

In [ ]:
# Creating a month categorical variable
# Create a string label for display
aw_all['month'] =  aw_all['StartDate'].dt.strftime('%b %Y') # Need .dt for series/panda 

# Certainly 1 way to get month from datetime but keep it as a datetime variable lol
aw_all['month_date'] =  pd.to_datetime(aw_all['month'], format='%b %Y') # Need .dt for series/panda

# Need to localize datetime to US/Eastern
aw_all['month_date'] = aw_all['month_date'].dt.tz_localize("US/Eastern")

# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = aw_all.sort_values('month_date')['month'].unique()
aw_all['month'] = pd.Categorical(aw_all['month'], categories=month_order, ordered=True)


# Creating a week categorical variable
# calculating out necessary time metrics when reading in apple workouts data
## calculates how many days each weekday is from monday and subtracts that from the orignal date to get back to start of the week Monday
aw_all['week_date'] = (
    aw_all['StartDate'] - pd.to_timedelta(aw_all['StartDate'].dt.weekday, unit="D") # dt.weekday uses Monday as 0 
).dt.normalize() # Resulting date is in EST (I believe this is the correct move when filtering later on)

    # Create a string label for display
aw_all['week'] = aw_all['week_date'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = aw_all.sort_values('week_date')['week'].unique()
aw_all['week'] = pd.Categorical(aw_all['week'], categories=week_order, ordered=True)

# DF and Visualizations pairs

### Frequency bar chart grouped by exercise type and month

In [ ]:
activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

df_counts = (aw_all[(aw_all['metric'] == 'Duration') & (aw_all['activity'].isin(activities)) & 
                    (aw_all['month_date'] > l_7_m)]
                    .groupby(['month_date', 'activity'], observed=True) # Include only combinations that actually occur in the data
                    .size() 
                    .reset_index(name='n'))

month_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='month_date',
    category_col='activity',
    value_cols=['n'])

month_count_data['n'] = month_count_data['n'].astype(int)  # Convert to int for better readability

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
month_count_data['activity'] = month_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

# Merging in categorical month label
month_count_data = pd.merge(month_count_data,
    aw_all[['month', 'month_date']].drop_duplicates(),
    on='month_date',
    how='left')

freq_cush = 2

session_y_max = int(month_count_data["n"].max() + freq_cush)

In [ ]:
############### Frequency bar chart grouped by exercise type and month ###############

plot = (ggplot(month_count_data, aes(x='month', y='n', fill='activity')) + 
geom_bar(stat='identity', position='dodge', color = "Black") +

geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 

scale_fill_manual(values= {"Swimming": "#3A8FD4",
                            "Cycling":  "#FF6B2B",
                            "Running":  "#7B5EA7", 
                            "Walking":  "#E63946",
                            "Weights":  "#2EAF6E"} ) +


scale_y_continuous(breaks = range(0, session_y_max + 1, 2),
                limits = [0, session_y_max]) +

labs(title='',
    x='',
    y='Sessions',
    fill='') +

workout_theme 
    )

plot

In [ ]:
save_plot(plot, os.path.join(output_path, "monthly_frequency.png") )

print(f"Monthly workout frequency graph generated and saved! {today.strftime('%m-%d-%Y')}")

### Frequency bar chart grouped by exercise type and week (Deprecated)

In [ ]:
%%skip
############### Frequency bar chart grouped by exercise type and week ###############

activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

df_counts = (aw_all[(aw_all['metric'] == 'Duration') & (aw_all['activity'].isin(activities)) & (aw_all['week_date'] > l_3_m)]
    .groupby(['week_date', 'activity'])
    .size()
    .reset_index(name='n'))

week_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='week_date',
    category_col='activity',
    value_cols=['n'])

week_count_data['n'] = week_count_data['n'].astype(int)  # Convert to int for better readability

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
week_count_data['activity'] = week_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

week_count_data = pd.merge(week_count_data,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
%%skip
############### Frequency bar chart grouped by exercise type and week ###############

plot = (ggplot(week_count_data, aes(x='week', y='n', fill='activity')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    
    scale_fill_brewer(type='qual', palette='Set2') +
    #scale_color_manual(values={'Running': 'Black', 'Cycling': 'Gray'}) +
    scale_y_continuous(breaks = range(0, 6),
                    limits = [0, 5]) +

    labs(title='Workout Frequency by Week and Activity',
        x='',
        y='Sessions',
        fill='Activity') +
    #theme_matplotlib() +
    theme_seaborn() +
    theme(figure_size=(10, 5)))

plot

### Distance per week grouped by exercise type

In [ ]:
############### Distance per week grouped by exercise type ###############

miles_week = (aw_all[(aw_all['activity'].isin(['Running', 'Cycling'])) & (aw_all['metric'].str.contains("Distance")) & (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_Miles='sum', n='count')  # compute both mean and count
    .round(2) # Round to 2 decimal places
    .reset_index())

# What's interesting is that if biking or running has 0 observations missing observations won't be filled in
full_miles_week = fill_missing_combinations(
    original_df=miles_week,
    aggregated_df=miles_week,
    time_col='week_date',
    category_col='activity',
    value_cols=['Total_Miles', 'n'])


full_miles_week = pd.merge(full_miles_week,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

# Ensuring that Total_Miles variable is a float
full_miles_week["Total_Miles"]  = round( full_miles_week["Total_Miles"].astype("float"), 1) 

# Using the max point in the graph to calculate max Y axis to give enough room for horizonal legend
y_max = full_miles_week["Total_Miles"].max()

cushion = 6

y_limit = round(cushion + y_max)

In [ ]:
############### Distance per week grouped by exercise type ###############

plot = (ggplot(full_miles_week, aes(x='week', y='Total_Miles', group='activity', color='activity')) +

geom_line(size=2) +
geom_point(size=2) +

# geom_bar(stat='identity', position='dodge', color = "Black") +

geom_text(aes(label='Total_Miles'), color = "black", va='bottom', nudge_y = .35) +  # position=position_dodge(width=.9),

scale_y_continuous(breaks=range(0, y_limit, 5),
                    # minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                    limits=[0, y_limit]) +

scale_color_manual(values={'Running': '#7B5EA7',
                            'Cycling': '#FF6B2B'}) +

labs(title="",
        x="",
        y="Miles",
        color="") +

workout_theme
)

plot

In [ ]:
save_plot(plot, os.path.join(output_path,"distance.png") )

print(f"Cycling and running distance graphs generated and saved! {today.strftime('%m-%d-%Y')}")

### Minutes per week grouped by cardio and weights (Deprecated)

In [ ]:
%%skip
############### Minutes per week grouped by cardio and weights ###############

mins_week = (aw_all[
        (aw_all['activity_type'].notna()) &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity_type', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())

full_mins_week = fill_missing_combinations(
    original_df=mins_week,
    aggregated_df=mins_week,
    time_col='week_date',
    category_col='activity_type',
    value_cols=['Total_min', 'n'])

full_mins_week = pd.merge(full_mins_week,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
%%skip
############### Minutes per week grouped by cardio and weights ############### 

plot = (ggplot(full_mins_week, aes(x='week', y='Total_min', color='activity_type', group='activity_type')) +
    geom_point(color = "Black", size = .6) +
    geom_line(size = 1.5) +

    geom_text(aes(label='n'), va='bottom', color = "black") + 
    #geom_bar(stat='identity', position='dodge', color = "Black") +
    #geom_text(aes(label='Total_min'), format_string='{:.1f}',position=position_dodge(width=.9), va='bottom') + #Format count?

    geom_hline(yintercept=150, color="#549f74", linetype='dashed', size=1) + # Cardio goal
    geom_hline(yintercept=80, color="#b36a62", linetype='dashed', size=1) + # Weights goal
    
    scale_y_continuous(breaks = range(0, y_max, 50),
                    #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, y_max]) +

    # Manual color scales
    #scale_fill_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +  # Bar colors

    scale_color_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +

    labs(title= "Minutes per Week by Activity",
        x="",
        y="Minutes", 
        fill="Activity") +
    
    theme_seaborn() +
    theme(figure_size=(10, 5),
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted"),
        legend_direction="vertical",
        legend_title=element_blank(), 
        legend_position=(.95, .92) ))

plot

### Workout time grouped by cardio activities and weights

In [ ]:
# Calculating workout during on activity types for their underlying activities 

## Cardio df
full_mins_cardio = (aw_all[
        (aw_all['activity_type'] == "Cardio") &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())


## Weight df
full_mins_weight = (aw_all[
        (aw_all['activity_type'] == "Weights") &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())

## Changing Total_min from float to rounded int
full_mins_cardio["Total_min"] = round(full_mins_cardio["Total_min"], 0).astype("int64")

## Offsetting the weekdate so I can plot both cardio and weight bars on the same graph
full_mins_cardio['x_nudged']  = full_mins_cardio['week_date']  - dt.timedelta(days=1)
full_mins_weight['x_nudged'] = full_mins_weight['week_date'] + dt.timedelta(days=1)

full_mins_weight["Total_min"] = round( full_mins_weight["Total_min"].astype("float"), 2)

In [ ]:
# Calculating total minutes per week for cardio workouts (used to set y axis max and label calculation)
total_mins = full_mins_cardio.groupby(['week_date'])['Total_min'].agg(week_min='sum')

# Instituing some quality of life for my legend
y_cush = 50 
y_max = int(y_cush + max(total_mins["week_min"]))

# Merging total cardio mins for each week to full_mins_cardio and renaming to cardio_labels 
cardio_labels = pd.merge(full_mins_cardio, total_mins, on= "week_date", how="left")

# Calculating the percetnage of cardio time spent on each activity for each week 
cardio_labels["percent_time"] = round(cardio_labels["Total_min"] / cardio_labels["week_min"] * 100, 2)

# Sorting cardio_labels by week_date and activity for cumsum calculation to match how activities are stacked on barplot
cardio_labels = cardio_labels.sort_values(by=['week_date', 'activity'], ascending=[True, False])
cardio_labels["cumsum_min"] = cardio_labels.groupby('week_date')['Total_min'].cumsum()

In [ ]:
############### Plot ###############
plot = (ggplot() +

    # Cardio stacked bar chart by activity subtypes (nudged to the left)
    geom_bar(
        data=full_mins_cardio,
        mapping=aes(x='x_nudged', y='Total_min', fill='activity'),
        stat='identity', position='stack', width=2, color="#45484B", size=0.7 ) +

    # Cardio Labels: minutes for each cardio subtype
    geom_text(
        data=cardio_labels,
        mapping=aes(x='x_nudged', y='cumsum_min', label='Total_min'),
        #format_string='{:.1f}', 
        va='bottom', color='black', size=10 ) +

    # Weights barchart, not stacked because only 1 activity type (nudged right)
    geom_bar(
        data=full_mins_weight,
        mapping=aes(x='x_nudged', y='Total_min', fill='activity'),
        stat='identity', position='stack', width=2, color='#45484B', size=0.7 ) +

    # Weight labels: count of weight exercises per week
    geom_text(data = full_mins_weight,
            mapping= aes(x='x_nudged', y='Total_min', label='n'), 
            va='bottom', color = "black", size = 10) +

    # 
    scale_fill_manual(values= {"Swimming": "#3A8FD4",
                            "Cycling":  "#FF6B2B",
                            "Running":  "#7B5EA7", 
                            "Walking":  "#E63946",
                            "TraditionalStrengthTraining":  "#2EAF6E"},
                            labels={"TraditionalStrengthTraining": "Weights"}) +

    # Setting goal lines for cardio and weight exercises per week
    geom_hline(yintercept=150, color="#b36a62", linetype='dashed', size=1) +
    geom_hline(yintercept=80,  color="#549f74", linetype='dashed', size=1) + 

    scale_y_continuous(breaks=range(0, y_max, 50), 
                    limits=[0, y_max]) +

    scale_x_datetime(date_breaks='1 week', 
                    date_labels='%b %d',
                    expand=(.01, 0))  +

    labs(title="", 
        x="", 
        y="Minutes", 
        fill="") +

    workout_theme
        )

plot

In [ ]:
save_plot(plot, os.path.join(output_path,"minutes.png") )

print(f"Minutes per exercise graph generated and saved! {today.strftime('%m-%d-%Y')}")

### Workout time by week over time

In [ ]:
# Gather average weekly time spent working out using last 3 months data
workout_time_df = aw_all[(aw_all['metric'] == 'Duration') & (aw_all['week_date'].dt.year >= 2024) ].groupby(['week_date'])['value'].agg(Time='sum', n='count').reset_index()

workout_time_df["Hours"] = (workout_time_df["Time"]/60).round(2)
workout_time_df["Year"] = pd.Categorical(workout_time_df['week_date'].dt.year.astype(str))
workout_time_df["Hours"] = (workout_time_df["Time"]/60).astype("float").round(2)
workout_time_df["month_day"] = workout_time_df['week_date'].dt.strftime('%m-%d')  # Extract month-day for alignment

# Create a reference date in a common year (e.g., 2024) for plotting
workout_time_df["plot_date"] = pd.to_datetime('2024-' + workout_time_df["month_day"])

# Creating normalized datetime for all years
# We set every year to 2000 so they align on the x-axis, but keep month/day
workout_time_df['norm_date'] = workout_time_df['week_date'].apply(lambda dt: dt.replace(year=2000))

#workout_time_df["Hours"] = round(workout_time_df["Hours"].astype("float"), 2)

# Dynamically calculating y axis max with some cushion room for labels
y_cush = 1
y_limit = round(y_cush + workout_time_df['Hours'].max())

In [ ]:
plot = (ggplot(workout_time_df, aes(x='plot_date', y='Hours', group='Year', color='Year')) +
    geom_line(size=1.4) +
    geom_point(color="Black", size=.8) +
    # geom_text(aes(label='Hours'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment

    geom_hline(yintercept=3, color="#549f74", linetype='dashed', size=1) +  # Cardio goal
    
    scale_color_manual(values={"2024": "#B8D2ED", "2025": "#4A90C4", "2026": "#0A2E4D"} ) +

                            # '2024': "#52be80", '2025': '#ec7063', '2026': "#3414B3"}) +

    scale_x_datetime(date_labels='%b', date_breaks='1 month',
                        expand=(0, 8, 0, 1)) +  # https://plotnine.org/reference/scale.html

    scale_y_continuous(breaks=range(0, y_limit + 1),
                        # Defining breaks of y axis (every number between 0 and 10 within the limit)
                        limits=[0, y_limit]) +  # Defining zoom of y axis

    labs(title='',
            x='',
            y='Hours',
            color='') +

    workout_theme
)

plot

In [ ]:
save_plot(plot, os.path.join(output_path, "weekly.png") )

print(f"Weekly workout time graph generated and saved! {today.strftime('%m-%d-%Y')}")

### Minutes per week for all exercises (line grpah - Deprecated)

In [ ]:
%%skip
############### Minutes per week for all exercises ############### 

workout_time = (
    aw_all[(aw_all['week_date'] > l_3_m) & (aw_all['metric'] == 'Duration')]
    .groupby('week_date', observed=True)['value']
    .agg(Time='sum', n='count')
    .round(2)
    .reset_index())

workout_time = pd.merge(workout_time,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
%%skip
############### Minutes per week for all exercises ############### 

plot = (ggplot(workout_time, aes(x='week', y='Time')) +
    geom_line(color = "blue", size = 1) +
    #geom_point(aes(size = "n"), alpha = 0.6, color = "blue") +
    #geom_text(aes(label='n'), format_string='{:.0f}', va='bottom') +
    
    labs(title= "Minutes per Week by Activity",
         x="Week",
         y="Minutes") +

    #scale_x_datetime(date_labels='%b %d', date_breaks='1 week') +
    
    theme_seaborn() +
    
    theme(panel_grid_minor_y = element_line(color = "gray", linetype = "dotted"),
        figure_size=(10, 5),
        axis_text_x=element_text(angle=25, hjust=1),
        legend_position='none',
        axis_ticks_minor_x=element_blank()))

plot

### Activity Treemap Distribution

In [ ]:
############### Activity Treemap ###############  

activites = ['Running', 'Cycling', 'TraditionalStrengthTraining', 'Swimming']

activity_distribution = (aw_all[
        (aw_all['metric'] == "Duration") & 
        (aw_all['activity'].isin(activites)) & 
        (aw_all['week_date'] >= l_3_m)] # Filtering by the last 3 months
    .groupby(['activity'])['value']
    .agg(n='count')
    .reset_index())
            
# Add percent of total column
total = activity_distribution['n'].sum()
activity_distribution['percent'] = activity_distribution['n'] / total

# Format labels as "Activity<br>Count (Percent)"
activity_distribution['label'] = activity_distribution.apply(lambda row: f"{row['activity']}<br>{row['n']} ({row['percent']:.1%})", axis=1)

In [ ]:
# Create treemap visual

fig = px.treemap(
    activity_distribution,
    path=['label'],         # use custom label for full text
    values='n',
    color='n',
    color_continuous_scale='Blues')

fig.update_traces(hovertemplate='%{label}')  
fig.update_layout(showlegend=False, coloraxis_showscale=False)

In [ ]:
fig.write_html(os.path.join(output_path, "activity_treemap.html"), full_html=True, include_plotlyjs=True)

print(f"Plotly treemap graph generated and saved! {today.strftime('%m-%d-%Y')}")

### Step count grouped by month boxplot

In [ ]:
# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    apple_steps = pd.read_sql_query(
        text("""SELECT type, sum(value) as 'value', date(startDate, 'localtime') as 'date', DATETIME(startDate, 'start of month') as 'month_date'
                FROM apple_data_raw
                WHERE type = 'StepCount' AND value IS NOT NULL and date(startDate, 'start of month','+1 month','-1 day') >= :t_filter
                GROUP BY date(startDate)
                ORDER BY 'date'"""), connection,
        dtype={"value": "int64"},
        params={"t_filter": l_1_y.astimezone(ZoneInfo("UTC"))},
        parse_dates=['date', 'month_date'])

# Localizing date and month_date to US/Eastern because already convert to localtime in the SQL query
apple_steps['date'] = apple_steps['date'].dt.tz_localize("US/Eastern")
apple_steps['month_date'] = apple_steps['month_date'].dt.tz_localize("US/Eastern")

# Creating a month categorical variable
apple_steps['month'] = apple_steps['date'].dt.strftime('%b %Y') # Need .dt for series/panda

# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = apple_steps.sort_values('month_date')['month'].unique()
apple_steps['month'] = pd.Categorical(apple_steps['month'], categories=month_order, ordered=True)

# Dynamically calculating y axis max with some cushion room for labels
step_cush = 2000

step_y_max = int(apple_steps["value"].max() + step_cush)

In [ ]:
############### Step count grouped by month ############### 
plot = (
    ggplot(apple_steps, aes(x='month', y='value', fill='month')) +

    geom_boxplot(color="black") +
    stat_summary(fun_data="mean_cl_boot", geom = "point", fill = "white", color = "red") +

    geom_hline(yintercept=7500, color="#549f74", linetype='dashed', size=1) + # Step goal

    labs(title="", 
        x="", 
        y="Steps") +

    scale_y_continuous(breaks = range(0, step_y_max , 5000),
                    minor_breaks=range(0, step_y_max , 2500),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, step_y_max]) +

    workout_theme +

    # Overriding legend element to exclude legend  
    theme(legend_position='none') 
)
plot

In [ ]:
save_plot(plot, os.path.join(output_path, "steps.png") )

print(f"Monthly step count graph generated and saved! {today.strftime('%m-%d-%Y')}")

### Generating main KPIs

In [ ]:
# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    # daw = distinct apple workouts
    daw = pd.read_sql_query( 
        text("""
            SELECT DATE(aw.startDate, 'localtime') as Date, aw.activity, aw.value 
            FROM apple_workouts aw 
            WHERE aw.metric = 'Duration' AND aw.value > 10
            ORDER BY aw.StartDate desc 
                """), connection,
        parse_dates=['Date'])
    
# Creating a week categorical variable (copied from inital aw_all completion)
# calculating out necessary time metrics when reading in apple workouts data
## calculates how many days each weekday is from monday and subtracts that from the orignal date to get back to start of the week Monday
daw['week_date'] = (daw['Date'] - pd.to_timedelta(daw['Date'].dt.weekday, unit="D") ).dt.normalize() # Resulting date is in EST (I believe this is the correct move when filtering later on) # dt.weekday uses Monday as 0 

# Create a string label for display
daw['week'] = daw['week_date'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = daw.sort_values('week_date')['week'].unique()
daw['week'] = pd.Categorical(daw['week'], categories=week_order, ordered=True)


# Gathering times for filtering
today = dt.datetime.now(tz=ZoneInfo("US/Eastern"))
this_month = today.strftime('%b %Y')
last_month = (today - relativedelta(months=1)).strftime('%b %Y')
current_year = int(today.strftime('%Y'))
duration_length = 10

# Gather names for displaying on front end
current_month_name = today.strftime('%B')
last_month_name = (today - relativedelta(months=1)).strftime('%B') 

# Gathering wokrout counts (classifying a true workout to be greater than 10 minutes long)
## Current Month Workout Count
wokrout_count_CM = len( daw[ (daw['Date'].dt.strftime('%b %Y') == this_month) ] )

## Last month Workout Count
workout_count_LM = len( daw[ (daw['Date'].dt.strftime('%b %Y') == last_month) ] )

## Current Year Workout Count
workout_count_year = len( daw[ (daw['Date'].dt.year == current_year) ] )

# Gather average weekly time spent working out using last 3 months data
workout_time_df = daw[ (daw['Date'].dt.date > l_3_m.date() ) ].groupby(['week_date'])['value'].agg(Time='sum', n='count').reset_index()
workout_time_hrs_avg = (workout_time_df["Time"].mean()/60).round(2) # Converting minutes to hours

In [ ]:
# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    # das_l3m = daily average steps last 3 months
    das_l3m = pd.read_sql_query( 
        text("""
        SELECT AVG(value) as 'value'
            FROM (SELECT DATE(adr.startDate, 'localtime') as Date, SUM(adr.value) as 'value' 
                    FROM apple_data_raw adr
                    WHERE type = 'StepCount' AND value IS NOT NULL AND date(startDate, 'localtime') >= :t_filter
                    GROUP BY date(startDate, 'localtime')
                    ) 
            """), connection,
        params={"t_filter": l_3_m} )   

steps_L3_mon = round(das_l3m["value"].iloc[0], 0).astype("int")

In [ ]:
# Saving output to csv instead of directly connecting to flask bceuase this will be run as separate workflow
asaved_df = pd.DataFrame(
    {
    'workout_count_year': workout_count_year,
    'workout_count_LM': workout_count_LM,
    'workout_count_CM': wokrout_count_CM,
    'current_month_name': current_month_name,
    'last_month_name': last_month_name,
    'workout_time_hrs_avg': workout_time_hrs_avg,
    'steps_L3_mon': steps_L3_mon
    }, index=[0] # No idea why, but this is needed
)

asaved_df.to_csv(os.path.join(output_path, "kpi_stats.csv"), index=False)

### Heart Rate graph

In [ ]:
# Gather list of all workouts to choose from
with engine.connect() as connection:
    workouts = pd.read_sql_query(
        text("""
            SELECT DISTINCT aw.workout_id, DATETIME(aw.StartDate, 'localtime') as date, replace(aw.activity, 'TraditionalStrengthTraining', 'Weights') AS activity, concat_ws(" ", strftime('%m-%d-%Y', aw.StartDate),  aw.activity  ) as workout_label
            FROM apple_workouts aw 
            WHERE aw.workout_id IS NOT NULL
            ORDER  BY  aw.StartDate desc
        """), connection,
        dtype={"workout_id": "int64"},
        parse_dates=['date'])

workout_list = workouts["workout_label"].to_list()

In [ ]:
# Building an example workflow of choosing a workout
select_workout = "08-11-2026 Running"

selected_workout_id = workouts[workouts["workout_label"] == select_workout][["workout_id"]].iloc[0, 0]

In [ ]:
# Grabbing the duration of selected workout in UTC, we'll convert to EST for display in the graph

with engine.connect() as connection:
    times = pd.read_sql_query(
        text("""
            SELECT aw.workout_id as id, aw.StartDate as start_date, aw.value 
            FROM apple_workouts aw 
            WHERE aw.metric = 'Duration' AND aw.workout_id = :selected_workout_id
        """), connection,
        dtype={"id": "int64"},
        params={"selected_workout_id": int(selected_workout_id)},
        parse_dates=['start_date'])

# Adding the duration to the start time to get workout end time
times["end_date"] = times["start_date"] + dt.timedelta(minutes = times["value"].iloc[0] )

In [ ]:
# Grabbing heart rate values for for duration of the workout
with engine.connect() as connection:
    hr_df = pd.read_sql_query(
        text("""
            SELECT adr.value as 'HeartRate', DATETIME(adr.startDate, 'localtime') as date 
            FROM apple_data_raw adr 
            WHERE adr.type = 'HeartRate' AND
                adr.startDate BETWEEN :t_low AND :t_upper
            ORDER BY adr.startDate
        """), connection,
        dtype={"HeartRate": "float64"},
        params={"t_low": str( times["start_date"].iloc[0] ), "t_upper": str( times["end_date"].iloc[0] ) },
        parse_dates=['date'])

hr_df["run_time"] = hr_df["date"] - hr_df["date"].iloc[0]

In [ ]:
# Graphing heart rate over time

plot = (ggplot(hr_df, aes('run_time', 'HeartRate')) +
        
geom_line(size=1.2, color='red') +
geom_point(size=1.2, color='black') +

# Horizontal lines for HR zones
geom_hline(yintercept=[130, 145, 160, 180], linetype='dashed', color=['green', 'yellow', 'orange', 'red'], size=0.5) +

scale_x_datetime(date_labels='%H:%M:%S') +

labs(title= "",
    x="",
    y="Miles",
    fill = "Activity") +

workout_theme 

)

plot

### Heart Rate kpis

In [ ]:
# calculating the time difference between each heart rate measurement 
hr_df["time_delta"] = hr_df["date"].diff()

def format_mmss(total_seconds):
    minutes, seconds = divmod(int(round(total_seconds)), 60)
    return f"{minutes}:{seconds:02d}" # MM:SS for display

# Total workout duration
total_seconds = hr_df["time_delta"].sum().total_seconds()
total_time = round(total_seconds / 60, 2)          # decimal minutes (kept for % math)
total_time_str = format_mmss(total_seconds) 

print(f"Total Time: {total_time_str}")

## Time in zone 2
zone2_seconds = hr_df.loc[hr_df["HeartRate"] <= 145, "time_delta"].sum().total_seconds()
zone2_time = round(zone2_seconds / 60, 2)
zone2_time_str = format_mmss(zone2_seconds)

print(f"{zone2_time_str} in zone 2")

## pct in zone 2
zone2_time_pct = round(zone2_time / total_time * 100, 2)

print(f"{zone2_time_pct}% of time in zone 2")

## Time above zone 2
zone2_above_seconds = hr_df.loc[hr_df["HeartRate"] > 145, "time_delta"].sum().total_seconds()
zone2_time_above = round(zone2_above_seconds / 60, 2)
zone2_time_above_str = format_mmss(zone2_above_seconds)

print(f"{zone2_time_above_str} spent above zone 2")

## pct above zone 2
zone2_time_above_pct = round(zone2_time_above / total_time * 100, 2)
print(f"{zone2_time_above_pct}% of time spent above zone 2")

In [ ]:
# Grabbing some other KPIs at the specfic workout level
with engine.connect() as connection:
    workout_kpis = pd.read_sql_query(
        text("""
            SELECT aw.metric, aw.value, aw.measurement_type, aw.activity_type 
            FROM apple_workouts aw 
            WHERE aw.workout_id = :selected_workout_id
        """), connection,
        #dtype={"id": "int64"},
        params={"selected_workout_id": int(selected_workout_id)}
        )

In [ ]:
# Duration
#duration = workout_kpis.loc[workout_kpis["metric"] == "Duration", "value"].sum()

distance = workout_kpis.loc[workout_kpis["metric"] == "DistanceWalkingRunning", "value"].sum()

if distance > 0:
    pace_seconds = (total_time * 60) / distance
    pace_str = format_mmss(pace_seconds)

    print(f"Pace: {pace_str} min/mile")

In [ ]:
zone2_seconds = hr_df.loc[hr_df["HeartRate"] <= 145, "time_delta"].sum().total_seconds()

In [ ]:
# Saving heart rate kpis to csv
hr_csv = pd.DataFrame({
    'total_time': total_time,
    'zone2_time': zone2_time,
    'zone2_time_pct': zone2_time_pct,
    'zone2_time_above': zone2_time_above,
    'zone2_time_above_pct': zone2_time_above_pct
}, index=[0])

hr_csv.to_csv(f"{output_path}/hr_kpis.csv", index=False)

### Food

In [ ]:
# Grabbing a list of food categories to choose from
with engine.connect() as connection:
    food_categories = pd.read_sql_query(
        text("""
            SELECT DISTINCT food_category
            FROM fact_food ff 
            WHERE food_category in ('Caffeine (mg)', 'Energy (kcal)', 'Alcohol (g)', 'Carbs (g)', 'Fiber (g)', 
                                    'Sugars (g)', 'Fat (g)', 'Cholesterol (mg)', 'Protein (g)')
            ORDER BY food_category
        """), connection)

food_categories = food_categories["food_category"].to_list()

In [ ]:
food_categories

In [ ]:
food_category = 'Protein (g)'

# Grabbing heart rate values for for duration of the workout
with engine.connect() as connection:
    food = pd.read_sql_query(
        text("""
            SELECT fd.food_date, ff.value as value
            FROM food_daily fd
            LEFT OUTER JOIN fact_food ff 
                ON ff.food_daily_id = fd.id
            WHERE fd.completed = 1 AND 
                ff.food_category = :food_category AND
                ff.meal_time = 'Total' AND
                fd.food_date >= date('now', '-45 days')
            ORDER BY fd.food_date DESC 
        """), connection,
        #dtype={"HeartRate": "float64"},
        params={"food_category": food_category },
        parse_dates=['food_date'])

In [ ]:
plot = (ggplot(food, aes('food_date', 'value')) +
        
geom_line(size=1.2, color='red') +
geom_point(size=1.2, color='black') +

geom_hline(yintercept = food["value"].mean()  ) +

#scale_x_datetime(date_labels='%H:%M:%S') +

labs(title= "",
    x="",
    y=f"{food_category}") +

theme_seaborn() 

)

plot